In [ ]:
import pandas as pd

# load metadata

df = pd.read_csv(
    "eldiario_metadata.csv",
    low_memory=False
)

print("=" * 80)

print("rows")

print(len(df))

print("\ncolumns")

for c in df.columns:
    print(c)

print("\n" + "=" * 80)

# missing values

summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:

        empty = (

            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()

        )

    summary.append({

        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)

    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print("\n" + "=" * 80)

# important fields

important = [

    "url",
    "category",
    "published",
    "title",
    "author",
    "text_preview"

]

print("\nimportant fields\n")

for field in important:

    if field in df.columns:

        print(f"{field:20} OK")

    else:

        print(f"{field:20} MISSING")

In [ ]:
df

In [ ]:
# filter articles about russia

russia_keywords = [

    r"\brusi(a|o|os|as)?\b",
    r"\bkreml(in)?\b",
    r"\bmosc[uú]\b",
    r"\bputin\b",
    r"\blavrov\b",
    r"\bpeskov\b",
    r"\bshoigu\b",
    r"\bgerasimov\b",
    r"\bmedv[eé]dev\b",
    r"\bzakharova\b",
    r"\bmishustin\b",
    r"\bgazprom\b",
    r"\brosneft\b",
    r"\brosatom\b",
    r"\btransneft\b",
    r"\blukoil\b",
    r"\bduma\b",
    r"\bcrimea\b",
    r"\bguerra de ucrania\b",
    r"\bguerra de agresi[oó]n\b"

]

pattern = "|".join(russia_keywords)

russia_df = df[

    (

        df["title"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

    |

    (

        df["text_preview"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            na=False
        )

    )

].copy()

print(len(russia_df))

In [ ]:
# compare filtered corpus with old high recall corpus

old_df = pd.read_csv(
    "eldiario_ukraine_russia_high_recall.csv"
)

old_urls = set(
    old_df["url"]
)

new_urls = set(
    russia_df["url"]
)

print(f"old corpus: {len(old_urls):,}")

print(f"new corpus: {len(new_urls):,}")

In [ ]:
old_df

In [ ]:
print(f"overlap: {len(old_urls & new_urls):,}")

print(f"only old: {len(old_urls - new_urls):,}")

print(f"only new: {len(new_urls - old_urls):,}")

In [ ]:
# save final corpus after the second keyword filtering

russia_df.to_csv(

    "eldiario_SECONDROUND_final_articles.csv",

    index=False,

    encoding="utf-8-sig"

)

print("\nfinal corpus saved")

print(len(russia_df))

In [ ]:
russia_df

In [ ]:
print(russia_df["image_urls"].iloc[0])

In [ ]:
import os

SOURCE_FOLDER = (
    "."
)

print(os.path.exists(SOURCE_FOLDER))

files = os.listdir(SOURCE_FOLDER)

print(len(files))

print(files[:20])

In [ ]:
import hashlib
import re

url = "https://static.eldiario.es/clip/843d322c-bdf6-4ded-966d-e611a00132fd_16-9-aspect-ratio_default_0.jpg"

print("md5:")
print(hashlib.md5(url.encode()).hexdigest() + ".jpg")

print()

m = re.search(
    r"/clip/([a-f0-9-]+)_",
    url
)

print("uuid:")

if m:
    print(m.group(1) + ".jpg")


In [ ]:
import hashlib
import pandas as pd

INPUT_FILE = (
    "eldiario_SECONDROUND_final_articles.csv"
)

OUTPUT_FILE = (
    "eldiario_SECONDROUND_final_articles.csv"
)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

filenames = []

for urls in df["image_urls"].fillna(""):

    article_files = []

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        filename = (

            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest()

            + ".jpg"

        )

        article_files.append(
            filename
        )

    filenames.append(
        "|".join(article_files)
    )

df["image_filenames"] = filenames

df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"

)

print("done")

print(df[[
    "image_urls",
    "image_filenames"
]].head())

In [ ]:
import os
import time
import hashlib
import requests
import pandas as pd

from tqdm import tqdm

# paths

INPUT_FILE = (
    "eldiario_SECONDROUND_final_articles.csv"
)

OUTPUT_FOLDER = (
    "eldiario_images_SECONDROUND"
)

FAILED_FILE = (
    "eldiario_failed_secondround.csv"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

MAX_RETRIES = 3

# load articles

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

# collect urls

image_urls = []

for urls in df["image_urls"].fillna(""):

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        image_urls.append(url)

image_urls = list(dict.fromkeys(image_urls))

print(f"images to check: {len(image_urls):,}")

downloaded = 0

already_exists = 0

failed = []

for url in tqdm(image_urls):

    filename = (
        hashlib.md5(
            url.encode("utf-8")
        ).hexdigest()
        + ".jpg"
    )

    filepath = os.path.join(
        OUTPUT_FOLDER,
        filename
    )

    if os.path.exists(filepath):

        already_exists += 1
        continue

    success = False

    for _ in range(MAX_RETRIES):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=60
            )

            if response.status_code == 200:

                with open(filepath, "wb") as f:

                    f.write(response.content)

                downloaded += 1

                success = True

                break

        except Exception:

            pass

        time.sleep(2)

    if not success:

        failed.append({

            "image_url": url,
            "filename": filename

        })

pd.DataFrame(
    failed
).to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"

)

print()

print(f"downloaded: {downloaded:,}")

print(f"already existed: {already_exists:,}")

print(f"failed: {len(failed):,}")

print()

print("failed file:")

print(FAILED_FILE)

In [ ]:
import pandas as pd

# load final corpus
df = pd.read_csv(
    "eldiario_SECONDROUND_final_articles.csv",
    low_memory=False
)
df 

In [ ]:
import pandas as pd
import hashlib

# ---------- load files ----------

final_df = pd.read_csv(
    "eldiario_SECONDROUND_final_articles.csv",
    low_memory=False
)

old_df = pd.read_csv(
    "eldiario_ukraine_russia_high_recall.csv",
    low_memory=False
)

images_df = pd.read_csv(
    "eldiario_ukraine_russia_with_images.csv",
    low_memory=False
)

# ---------- add combined text ----------

final_df = final_df.merge(
    old_df[["url", "combined_text", "text_length"]],
    on="url",
    how="left"
)

# ---------- add image urls ----------

final_df = final_df.merge(
    images_df[["url", "image_urls"]],
    on="url",
    how="left"
)

# ---------- generate image filenames ----------

filenames = []

for urls in final_df["image_urls"].fillna(""):

    article_files = []

    for url in str(urls).split("|"):

        url = url.strip()

        if not url.startswith("http"):
            continue

        article_files.append(
            hashlib.md5(
                url.encode("utf-8")
            ).hexdigest() + ".jpg"
        )

    filenames.append("|".join(article_files))

final_df["image_filename"] = filenames

# ---------- final dataset ----------

dataset = final_df.rename(columns={
    "published": "publish_date",
    "combined_text": "text",
    "image_urls": "image_url"
})

dataset = dataset[
    [
        "category",
        "url",
        "title",
        "publish_date",
        "text",
        "image_url",
        "image_filename",
    ]
]


In [ ]:
dataset

In [ ]:
# save
dataset.to_csv(
    "eldiario_SECONDROUND_final_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

print(dataset.shape)
print(dataset.columns.tolist())

In [ ]:
import os
import time
import hashlib
import requests
import pandas as pd
from tqdm import tqdm

FINAL_FILE = "eldiario_SECONDROUND_final_articles.csv"
OLD_IMAGES_FILE = "eldiario_ukraine_russia_with_images.csv"
IMAGE_FOLDER = "eldiario_images_SECONDROUND"
FAILED_FILE = "eldiario_failed_missing.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

MAX_RETRIES = 3

final_df = pd.read_csv(FINAL_FILE, low_memory=False)
old_images_df = pd.read_csv(OLD_IMAGES_FILE, low_memory=False)

merged = final_df.merge(old_images_df[["url", "image_urls"]], on="url", how="left")

image_urls = []
for urls in merged["image_urls"].fillna(""):
    for url in str(urls).split("|"):
        url = url.strip()
        if url.startswith("http"):
            image_urls.append(url)

image_urls = list(dict.fromkeys(image_urls))

downloaded = 0
already_exists = 0
failed = []

for url in tqdm(image_urls):
    filename = hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg"
    filepath = os.path.join(IMAGE_FOLDER, filename)

    if os.path.exists(filepath):
        already_exists += 1
        continue

    success = False
    for _ in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=HEADERS, timeout=60)
            if response.status_code == 200:
                with open(filepath, "wb") as f:
                    f.write(response.content)
                downloaded += 1
                success = True
                break
        except Exception:
            pass
        time.sleep(2)

    if not success:
        failed.append({"image_url": url, "filename": filename})

pd.DataFrame(failed).to_csv(FAILED_FILE, index=False, encoding="utf-8-sig")

print(f"downloaded: {downloaded}")
print(f"already existed: {already_exists}")
print(f"failed: {len(failed)}")

In [ ]:
import pandas as pd
import hashlib

FINAL_FILE = "eldiario_SECONDROUND_final_articles.csv"
OLD_IMAGES_FILE = "eldiario_ukraine_russia_with_images.csv"

final_df = pd.read_csv(FINAL_FILE, low_memory=False)
old_images_df = pd.read_csv(OLD_IMAGES_FILE, low_memory=False)

merged = final_df.merge(old_images_df[["url", "image_urls"]], on="url", how="left")

filenames = []
for urls in merged["image_urls"].fillna(""):
    article_files = []
    for url in str(urls).split("|"):
        url = url.strip()
        if url.startswith("http"):
            article_files.append(hashlib.md5(url.encode("utf-8")).hexdigest() + ".jpg")
    filenames.append("|".join(article_files))

merged["image_filename"] = filenames
merged["image_url"] = merged["image_urls"]

final_df["image_url"] = merged["image_url"]
final_df["image_filename"] = merged["image_filename"]

In [ ]:
final_df.to_csv(FINAL_FILE, index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import ast
import os
import shutil

FINAL_FILE = "eldiario_SECONDROUND_final_articles.csv"
IMAGE_FOLDER = "eldiario_images_SECONDROUND"
EXTRA_FOLDER = "eldiario_images_extra"

os.makedirs(EXTRA_FOLDER, exist_ok=True)

df = pd.read_csv(FINAL_FILE, low_memory=False)

def parse_filenames(x):
    if pd.isna(x):
        return []
    return [f.strip() for f in str(x).split("|") if f.strip()]

needed_filenames = set(f for row in df["image_filename"].apply(parse_filenames) for f in row)

folder_files = set(os.listdir(IMAGE_FOLDER))
to_move = folder_files - needed_filenames

print("Files to move:", len(to_move))

for filename in to_move:
    shutil.move(
        os.path.join(IMAGE_FOLDER, filename),
        os.path.join(EXTRA_FOLDER, filename)
    )

print("done")

In [ ]:
final_df